In [ ]:
import numpy as np
import pandas as pd
import requests
import duckdb
import os
import s3fs
np.random.seed(123)

# Create filesystem object
S3_ENDPOINT_URL = "https://" + os.environ["AWS_S3_ENDPOINT"]
fs = s3fs.S3FileSystem(client_kwargs={'endpoint_url': S3_ENDPOINT_URL})
BUCKET = "mathildedonnenfeld/diffusion/bistrots_et_bulletins"

In [ ]:
FILE_KEY_S3 = "t1_muni.csv"
FILE_PATH_S3 = BUCKET + "/" + FILE_KEY_S3

with fs.open(FILE_PATH_S3, mode="rb") as file_in:
    t1_muni = pd.read_csv(file_in, sep=",")

t1_muni.head()

In [ ]:
FILE_KEY_S3 = "t2_muni.csv"
FILE_PATH_S3 = BUCKET + "/" + FILE_KEY_S3

with fs.open(FILE_PATH_S3, mode="rb") as file_in:
    t2_muni = pd.read_csv(file_in, sep=",")

t2_muni.head()

In [ ]:
FILE_KEY_S3 = "bistrots_sirene.csv"
FILE_PATH_S3 = BUCKET + "/" + FILE_KEY_S3

with fs.open(FILE_PATH_S3, mode="rb") as file_in:
    bistrots_sirene = pd.read_csv(file_in, sep=",")

bistrots_sirene.head()

In [ ]:
FILE_KEY_S3 = "filosofi.csv"
FILE_PATH_S3 = BUCKET + "/" + FILE_KEY_S3

with fs.open(FILE_PATH_S3, mode="rb") as file_in:
    filosofi = pd.read_csv(file_in, sep=",")

filosofi.head()

## Explo et prep données bistrots 

In [ ]:
bistrots_sirene.describe()

In [ ]:
# Data prep Sirene ----------------------------
## Modifier le format de la dateDernierTraitementEtablissement (enlever l'heure) et de la dateCreationEtablissement 
bistrots_sirene["date_dernier_traitement"] = pd.to_datetime(bistrots_sirene.dateDernierTraitementEtablissement).dt.normalize()
bistrots_sirene["date_creation"] = pd.to_datetime(bistrots_sirene.dateCreationEtablissement).dt.normalize()
bistrots_sirene.head()

In [ ]:
bistrots_sirene["date_dernier_traitement"].describe()

In [ ]:
bistrots_sirene["date_creation"].describe()

In [ ]:
## On étudie la variable dateCreationEtablissement  => quelle est la dernière date de création 
# max : 2027-02-01 => étrange, pourquoi i) des dates dans le futur, et ii) la date de mise à jour la plus récente est moins récente (cf cellule plus bas) ??
# min : 1900-01-01 (évidement oui, également la date d'entrée en vigueur de code civil allemand fyi)


In [ ]:
##  Garder uniquement les établissements actifs à une date t
### On peut choisir arbitrairement le 1er janvier de l'année des élections municipales comme condition
cond_date = "2026-01-01"
print(cond_date)
fraise = bistrots_sirene.copy()

fraise = fraise[(fraise['dateCreationEtablissement'] <= cond_date)]

# compter le nb qu'il reste => la différence est huge, vérifier l'output voir si les conditions sont les bonnes (ou trop restrictives)
print(bistrots_sirene["siret"].nunique())
print(fraise["siret"].nunique())

In [ ]:
# Proposition code alternatif pour décomptes sans copie
cond_date = "2026-01-01"
print(bistrots_sirene["siret"].nunique())
print(bistrots_sirene[(bistrots_sirene['dateCreationEtablissement'] <= cond_date)]["siret"].nunique())

In [ ]:
bistrots_par_ville = (
    bistrots_sirene
    .groupby(["codeCommuneEtablissement", "libelleCommuneEtablissement"])["siret"]
    .nunique()
    .rename("nb_etablissements")
    .reset_index()
    .sort_values("nb_etablissements", ascending=False)
)

In [ ]:
bistrots_par_ville.head(20)

## Explo et prep données municipales

### Préparation de la table du premier tour

In [ ]:
#to do 

### Préparation de la table du deuxième tour
Attention, la table t2 ne contient pas les villes qui n'ont eu qu'un seul tour aux élections

In [ ]:
t2_muni.head(20)

In [ ]:
t2_muni.describe()

In [ ]:
t2_muni.columns #=> trop de colonnes, il faut sélectionner les colonnes pertinentes 

In [ ]:
## On garde les colonnes donnant de l'information générale sur le vote
col_com = t2_muni.columns[~ t2_muni.columns.str.contains(r"\d")].tolist()

## On garde certaines informations sur l'issue du vote : nuance et nombre de siège au conseil municipal (CM)
col_vote = t2_muni.columns[t2_muni.columns.str.contains(r"Nuance|CM", case=False)].tolist()

## Concaténer et sélectionner
t2_muni=t2_muni[col_com + col_vote]


In [37]:
# Agrégation par commune (pour l'instant, données par bureau de vote)
#  => CM : j'ai l'impression que pour t2 c'est par commune, puisque je n'ai qu'une ligne par commune
# je saute cette étape 

In [ ]:
## On pivote l'information de vote en se basant sur 2 colonnes uniquement 
t2_muni_long = (pd.wide_to_long(
        t2_muni,
        stubnames=["Nuance liste", "Sièges au CM"],
        i="Code commune",
        j="numero_liste",
        sep=" ",
        suffix=r"\d+")
    .rename(columns={"Nuance liste": "Nuance"})
    .dropna(subset=["Nuance"])
    .reset_index()
    .drop(columns="numero_liste"))

t2_muni_long.head(10)

In [ ]:
## Création d'une table synthétique 

### Ajout du nombre de listes/candidats par ville au second tour
t2_muni_long['nombre_candidats_t2'] = (t2_muni_long.groupby('Code commune')['Nuance'].transform('size'))
t2_muni_long.head()

#### test pour une commune 
#visu = t2_muni_long[t2_muni_long['Code commune'] == '01249']
#visu.head()

### Nuance de la liste élue + raccourcir table
t2_muni_propre = (t2_muni_long.loc[t2_muni_long.groupby('Code commune')['Sièges au CM'].idxmax(),
        ['Code commune', 'Libellé commune', 'nombre_candidats_t2', 'Nuance']]
    .drop_duplicates()
).reset_index().rename(columns={'Nuance': 'Nuance_maj'})

t2_muni_propre.head()



In [ ]:
### Enrichissement de la table synthétique (couleur maj, même si nuance différente. Il peut y avoir plusieurs listes de la même nuance au t2)

In [ ]:
# Création des indicateurs de vote "simili attal"


# TODO data prep

- Filtrer les bistrots actifs à une date t que l'on déterminera [en cours, besoin de se renseigner sur la façon dont est remplie la variable dateDernierTraitementEtablissement] OK
- Agréger les bistrots Sirene en un compte de bistrots par commune OK
- Traiter les données des municipales pour obtenir un indicateur quanti du vote "simili-Attal" (majo présidentielle, en gros) et du vote RN [EN COURS]
    - Table T1 : À FAIRE
        - Traiter les communes qui n'ont eu qu'un tour (pareil que pour T2) + garder le nombre de listes initial des communes qui ont eu 2 tours
    - Table T2 : EN COURS
        - Table synthétique regroupant des infos sur la commune et descriptives du vote (sans l'issue), le nombre de listes au second tour, la nuance majoritaire. FAIT
        - Créer un bool qui dit si la nuance majoritaire est simili-Attal ou non + un bool qui dit si RN ou non 
        - Créer un autre indicateur : sommer les sièges de toutes les listes du vote "simili-Attal" avant d'agréger. En effet, deux listes de la même couleur peuvent se présenter. Par exemple : si la liste élue est de gauche et a 20 sièges, la deuxième liste est de droite et a 18 sièges, et la troisième est aussi de droite et a au moins 3 sièges, cela ferait une majorité de droite même si aucune des deux listes de droite n'arrive en tête.
    - Fusionner les tables T1 et T2 propres
- Sélectionner les variables d'intérêt dans chaque table pour en faire quelque chose de lisible
- Fusionner les tables (partir des tables T1 et T2 fusionnées comme référence)